In [ ]:
# Cell 1: Install required packages

!pip install requests beautifulsoup4 lxml python-dotenv openai pandas

In [2]:
# Cell 2: Import required libraries

import os
import re
import time
import json
import pandas as pd
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from dotenv import load_dotenv

from openai import OpenAI

In [5]:
# Cell 3: Fetch website content
# King-mode instructions:
# 1. Use a valid company website URL.
# 2. Always include https:// at the beginning.
# 3. This function fetches raw HTML from the website.
# 4. It also handles errors safely.
# 5. If the website blocks scraping, it will show a clear error message.

# website_url = "https://example.com"  # Replace this with your company website URL

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}


def fetch_website_content(url):
    """
    Fetch raw HTML content from a website URL.

    Args:
        url (str): Company website URL.

    Returns:
        str: Raw HTML content if successful, otherwise None.
    """

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        print("Website content fetched successfully.")
        print("Status Code:", response.status_code)
        print("Content Length:", len(response.text))

        return response.text

    except requests.exceptions.Timeout:
        print("Error: Website request timed out.")
        return None

    except requests.exceptions.ConnectionError:
        print("Error: Could not connect to the website.")
        return None

    except requests.exceptions.HTTPError as e:
        print("HTTP Error:", e)
        return None

    except requests.exceptions.RequestException as e:
        print("Request Error:", e)
        return None


# homepage_html = fetch_website_content(website_url)

In [29]:
# Cell 4: Fetch website links
# King-mode instructions:
# 1. This cell extracts all links from the homepage HTML.
# 2. It converts relative links like "/about" into full URLs.
# 3. It removes duplicate links.
# 4. It removes useless links like mailto, tel, javascript, and page anchors.
# 5. It keeps only internal company website links.

def get_domain(url):
    """
    Extract domain name from website URL.
    Example:
    https://example.com/about -> example.com
    """
    parsed_url = urlparse(url)
    return parsed_url.netloc.replace("www.", "")


def clean_url(url):
    """
    Remove URL fragments like #section.
    """
    parsed = urlparse(url)
    cleaned = parsed._replace(fragment="").geturl()
    return cleaned.rstrip("/")


def fetch_website_links(html, base_url):
    """
    Extract all internal website links from homepage HTML.

    Args:
        html (str): Raw HTML content.
        base_url (str): Main company website URL.

    Returns:
        list: Clean internal website links.
    """

    if html is None:
        print("No HTML content found. Please run Cell 3 successfully first.")
        return []

    soup = BeautifulSoup(html, "lxml")
    base_domain = get_domain(base_url)

    links = []

    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"].strip()

        # Skip useless links
        if (
            not href
            or href.startswith("#")
            or href.startswith("mailto:")
            or href.startswith("tel:")
            or href.startswith("javascript:")
        ):
            continue

        # Convert relative URL to full URL
        full_url = urljoin(base_url, href)

        # Clean URL
        full_url = clean_url(full_url)

        # Keep only internal company links
        link_domain = get_domain(full_url)

        if link_domain == base_domain:
            links.append(full_url)

    # Remove duplicates and sort
    unique_links = sorted(list(set(links)))

    print("Total internal links found:", len(unique_links))

    return unique_links


# all_links = fetch_website_links(homepage_html, website_url)

# Display links in table format
# links_df = pd.DataFrame(all_links, columns=["Website Links"])
# links_df

In [8]:
# Cell 5: Link Selection System Prompt
# King-mode instructions:
# 1. This prompt tells the LLM to act like a business website link analyzer.
# 2. The LLM must review all extracted website links.
# 3. The LLM should select only useful links for creating a business brochure.
# 4. The output must be valid JSON format.
# 5. Each selected link should include page type, page name, URL, and reason.

link_system_prompt = """
You are an expert business website analyst.

Your task is to analyze a company's website links and select the most useful links for creating a professional business brochure.

You must identify important business pages such as:

- Home page
- About page
- Services page
- Products page
- Solutions page
- Industries page
- Case studies page
- Customers page
- Pricing page
- Contact page
- Career page
- Team page
- Mission or values page

Ignore links that are not useful for a business brochure, such as:

- Login pages
- Signup pages
- Privacy policy
- Terms and conditions
- Cookie policy
- Social media links
- Blog tags
- Search pages
- Cart pages
- Account pages
- Duplicate links
- Empty or broken links

Return only valid JSON.

The JSON format must look like this:

{
  "selected_links": [
    {
      "page_type": "home_page",
      "page_name": "Home Page",
      "url": "https://example.com",
      "reason": "This page gives the main overview of the company."
    },
    {
      "page_type": "about_page",
      "page_name": "About Page",
      "url": "https://example.com/about",
      "reason": "This page explains the company background, mission, and values."
    },
    {
      "page_type": "services_page",
      "page_name": "Services Page",
      "url": "https://example.com/services",
      "reason": "This page explains the services offered by the company."
    },
    {
      "page_type": "career_page",
      "page_name": "Career Page",
      "url": "https://example.com/careers",
      "reason": "This page may show company culture, growth, and team values."
    },
    {
      "page_type": "contact_page",
      "page_name": "Contact Page",
      "url": "https://example.com/contact",
      "reason": "This page provides contact information for the brochure call to action."
    }
  ]
}

Important rules:
- Return JSON only.
- Do not write explanations outside the JSON.
- Do not include irrelevant links.
- Do not create fake URLs.
- Use only the links provided by the user.
- Select a maximum of 10 to 12 useful links.
"""

In [15]:
# Cell 6: Create Link Selection User Prompt
# King-mode instructions:
# 1. This function creates the user prompt for the LLM.
# 2. It accepts the company website URL as an attribute.
# 3. It also accepts all extracted website links.
# 4. The LLM will use this prompt to select only relevant brochure links.
# 5. The output should match the JSON format from the system prompt.

def create_link_user_prompt(website_url, fetched_website_urls):
    """
    Create user prompt for selecting relevant brochure links.

    Args:
        website_url (str): Main company website URL.
        all_links (list): List of all extracted website links.

    Returns:
        str: User prompt for the LLM.
    """

    links_text = "\n".join([f"- {link}" for link in all_links])

    user_prompt = f"""
Company Website URL:
{website_url}

All Extracted Website Links:
{links_text}

Your task:
Analyze the above website links and select only the most relevant links for creating a professional business brochure.

Return the selected links in valid JSON format only.

Each selected link must include:
- page_type
- page_name
- url
- reason

Example page types:
- home_page
- about_page
- services_page
- products_page
- solutions_page
- industries_page
- case_studies_page
- customers_page
- pricing_page
- career_page
- team_page
- contact_page

Important:
- Use only the URLs provided above.
- Do not create fake URLs.
- Do not include privacy policy, terms, login, signup, or unrelated pages.
- Select maximum 10 to 12 links.
"""
    return user_prompt

    

In [25]:
load_dotenv(override=True)

client = OpenAI(base_url=os.getenv("OLLAMA_OPENAI_BASE_URL"), api_key=os.getenv("MY_API_KEY"))

model = os.getenv("DEFAULT_MODEL")

In [26]:
def select_relevant_links(url):
    
    link_user_prompt = create_link_user_prompt(
        website_url=url,
        fetched_website_urls=all_links
    )

    messages = [
        {
            "role": "system",
            "content": link_system_prompt
        },
        {
            "role": "user",
            "content": link_user_prompt
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
        max_tokens=2000,
        top_p=1
        # remove response_format for now
    )
    
    content = response.choices[0].message.content

    print("Raw LLM Response:")
    print(repr(content))

    if content is None or content.strip() == "":
        print("LLM returned empty response.")
        return None

    cleaned_content = content.strip()
    cleaned_content = cleaned_content.replace("```json", "")
    cleaned_content = cleaned_content.replace("```", "")
    cleaned_content = cleaned_content.strip()

    try:
        links = json.loads(cleaned_content)
        return links

    except json.JSONDecodeError:
        print("Response is not valid JSON.")
        print("Cleaned response:")
        print(cleaned_content)
        return None

In [27]:
selected_link_data = select_relevant_links("https://huggingface.co/")
selected_link_data

Raw LLM Response:
'{\n  "selected_links": []\n}'


{'selected_links': []}

In [30]:
len(all_links)

0

In [31]:
all_links[:20]

[]

In [34]:
website_url = "https://huggingface.co/"
homepage_html = fetch_website_content(website_url)
all_links = fetch_website_links(homepage_html, website_url)

len(all_links), all_links[::]

Website content fetched successfully.
Status Code: 200
Content Length: 146401
Total internal links found: 55


(55,
 ['https://huggingface.co',
  'https://huggingface.co/Intel',
  'https://huggingface.co/SeeSee21/Z-Anime',
  'https://huggingface.co/SulphurAI/Sulphur-2-base',
  'https://huggingface.co/Writer',
  'https://huggingface.co/allenai',
  'https://huggingface.co/amazon',
  'https://huggingface.co/blog',
  'https://huggingface.co/brand',
  'https://huggingface.co/changelog',
  'https://huggingface.co/chat',
  'https://huggingface.co/datasets',
  'https://huggingface.co/datasets/Jackrong/GLM-5.1-Reasoning-1M-Cleaned',
  'https://huggingface.co/datasets/Roman1111111/claude-opus-4.6-10000x',
  'https://huggingface.co/datasets/nvidia/Nemotron-Image-Training-v3',
  'https://huggingface.co/datasets/nvidia/Nemotron-Personas-Korea',
  'https://huggingface.co/datasets/open-thoughts/AgentTrove',
  'https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro',
  'https://huggingface.co/docs',
  'https://huggingface.co/docs/accelerate',
  'https://huggingface.co/docs/datasets',
  'https://huggingface.co/docs

In [33]:
selected_link_data = select_relevant_links("https://huggingface.co/")
selected_link_data

Raw LLM Response:
'{\n  "selected_links": [\n    {\n      "page_type": "home_page",\n      "page_name": "Home Page",\n      "url": "https://huggingface.co",\n      "reason": "Provides the main overview of Hugging Face, its platform and key offerings."\n    },\n    {\n      "page_type": "about_page",\n      "page_name": "Brand Page",\n      "url": "https://huggingface.co/brand",\n      "reason": "Describes the company story, mission, values and brand identity."\n    },\n    {\n      "page_type": "services_page",\n      "page_name": "Enterprise Services",\n      "url": "https://huggingface.co/enterprise",\n      "reason": "Details Hugging Face\'s enterprise‑grade services, support and solutions for businesses."\n    },\n    {\n      "page_type": "products_page",\n      "page_name": "Models Hub",\n      "url": "https://huggingface.co/models",\n      "reason": "Showcases the catalog of pre‑trained models that constitute a core product."\n    },\n    {\n      "page_type": "products_page",\n

{'selected_links': [{'page_type': 'home_page',
   'page_name': 'Home Page',
   'url': 'https://huggingface.co',
   'reason': 'Provides the main overview of Hugging Face, its platform and key offerings.'},
  {'page_type': 'about_page',
   'page_name': 'Brand Page',
   'url': 'https://huggingface.co/brand',
   'reason': 'Describes the company story, mission, values and brand identity.'},
  {'page_type': 'services_page',
   'page_name': 'Enterprise Services',
   'url': 'https://huggingface.co/enterprise',
   'reason': "Details Hugging Face's enterprise‑grade services, support and solutions for businesses."},
  {'page_type': 'products_page',
   'page_name': 'Models Hub',
   'url': 'https://huggingface.co/models',
   'reason': 'Showcases the catalog of pre‑trained models that constitute a core product.'},
  {'page_type': 'products_page',
   'page_name': 'Datasets Hub',
   'url': 'https://huggingface.co/datasets',
   'reason': 'Provides access to a large collection of datasets, another key p

In [35]:
# Cell 8: Fetch relevant links website content
# King-mode instructions:
# 1. This function takes selected relevant links from the LLM.
# 2. It visits each selected URL.
# 3. It fetches the website HTML content.
# 4. It cleans the HTML and extracts readable text.
# 5. It stores page_type, page_name, url, reason, and content.
# 6. This final content will be used later for the brochure LLM prompt.

def clean_website_text(html):
    """
    Clean raw HTML and extract readable website text.
    """

    if html is None:
        return ""

    soup = BeautifulSoup(html, "lxml")

    # Remove useless HTML sections
    for tag in soup(["script", "style", "noscript", "svg", "form"]):
        tag.decompose()

    # Remove common repeated layout sections
    for tag in soup.find_all(["nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator=" ")

    # Clean extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def get_relevant_links_content(selected_links_data):
    """
    Fetch and clean content from LLM-selected relevant links.

    Args:
        selected_links_data (dict): JSON data returned by LLM with selected_links.

    Returns:
        list: Relevant pages with cleaned website content.
    """

    relevant_pages_content = []

    if selected_links_data is None:
        print("No selected links data found.")
        return relevant_pages_content

    selected_links = selected_links_data.get("selected_links", [])

    if len(selected_links) == 0:
        print("No relevant links found inside selected_links_data.")
        return relevant_pages_content

    for item in selected_links:
        page_type = item.get("page_type", "")
        page_name = item.get("page_name", "")
        url = item.get("url", "")
        reason = item.get("reason", "")

        if not url:
            continue

        print(f"Fetching content from: {page_name} -> {url}")

        html = fetch_website_content(url)
        clean_text = clean_website_text(html)

        relevant_pages_content.append({
            "page_type": page_type,
            "page_name": page_name,
            "url": url,
            "reason": reason,
            "content": clean_text
        })

        # Small delay to avoid hitting website too fast
        time.sleep(1)

    print("Total relevant pages content fetched:", len(relevant_pages_content))

    return relevant_pages_content

In [36]:
relevant_pages_content = get_relevant_links_content(selected_link_data)

relevant_pages_content

Fetching content from: Home Page -> https://huggingface.co
Website content fetched successfully.
Status Code: 200
Content Length: 146401
Fetching content from: Brand Page -> https://huggingface.co/brand
Website content fetched successfully.
Status Code: 200
Content Length: 32133
Fetching content from: Enterprise Services -> https://huggingface.co/enterprise
Website content fetched successfully.
Status Code: 200
Content Length: 159335
Fetching content from: Models Hub -> https://huggingface.co/models
Website content fetched successfully.
Status Code: 200
Content Length: 307473
Fetching content from: Datasets Hub -> https://huggingface.co/datasets
Website content fetched successfully.
Status Code: 200
Content Length: 220184
Fetching content from: Spaces -> https://huggingface.co/spaces
Website content fetched successfully.
Status Code: 200
Content Length: 371876
Fetching content from: Blog -> https://huggingface.co/blog
Website content fetched successfully.
Status Code: 200
Content Lengt

[{'page_type': 'home_page',
  'page_name': 'Home Page',
  'url': 'https://huggingface.co',
  'reason': 'Provides the main overview of Hugging Face, its platform and key offerings.',
  'content': 'Hugging Face – The AI community building the future. The AI community building the future. The platform where the machine learning community collaborates on models, datasets, and applications. Explore AI Apps or Browse 2M+ models Trending on this week Models Updated about 20 hours ago • 787k • 3.65k Updated about 24 hours ago • 55.5k • 295 Updated 14 days ago • 155k • 1.33k Updated 2 days ago • 16.6k • 281 Updated 10 days ago • 3.82k • 191 Browse 2M+ models Spaces Wan2.2 14B Fast Preview 🐌 966 generate a video from an image with a text prompt Qwen Image Edit + Loras built-in 👀 218 Qwen image edit with 🔞 loras FireRed Image Edit 1.0 Fast 🌖 1.16k FireRed-Image-Edit × Qwen-Image-Edit-Rapid (Transformers) Wan2.2 14B Preview 🐌 2.54k generate a video from an image with a text prompt The ultimate gui

In [37]:
# Cell 9: Brochure System Prompt and User Prompt
# King-mode instructions:
# 1. The system prompt tells the LLM to act like a professional brochure writer.
# 2. The user prompt gives the company website URL, relevant links, and page content.
# 3. The LLM must create a clean, professional business brochure.
# 4. The LLM should use only the provided website content.
# 5. The LLM should not create fake company details.

brochure_system_prompt = """
You are a professional business brochure writer, marketing strategist, and brand communication expert.

Your task is to create a clear, attractive, and professional business brochure using only the provided company website links and website content.

You must write the brochure in a business-friendly tone that is easy to understand.

Important rules:
- Use only the provided website content.
- Do not create fake company details.
- Do not invent services, products, pricing, locations, awards, or contact details.
- If some information is missing, write the brochure in a general but honest way.
- Make the brochure suitable for customers, partners, investors, and business audiences.
- Keep the language polished, simple, and professional.
- Organize the brochure with clear headings and sections.
"""

In [38]:
# Cell 10: Create Brochure User Prompt

def create_brochure_user_prompt(website_url, relevant_pages_content):
    """
    Create user prompt for generating a business brochure.

    Args:
        website_url (str): Main company website URL.
        relevant_pages_content (list): List of selected relevant pages with cleaned content.

    Returns:
        str: User prompt for brochure generation.
    """

    pages_text = ""

    for index, page in enumerate(relevant_pages_content, start=1):
        page_type = page.get("page_type", "")
        page_name = page.get("page_name", "")
        url = page.get("url", "")
        reason = page.get("reason", "")
        content = page.get("content", "")

        # Limit each page content to avoid sending too many tokens
        content = content[:4000]

        pages_text += f"""
Page {index}
Page Type: {page_type}
Page Name: {page_name}
URL: {url}
Reason this page is useful: {reason}

Website Content:
{content}

--------------------------------------------------
"""

    brochure_user_prompt = f"""
Create a professional business brochure for the company using the relevant website links and website content below.

Main Company Website:
{website_url}

Relevant Website Pages and Content:
{pages_text}

The brochure must include these sections:

1. Company Name
2. Business Tagline
3. Company Overview
4. What the Company Does
5. Products or Services
6. Key Features
7. Business Benefits
8. Target Customers or Industries
9. Why Choose This Company
10. Trust Signals or Strengths
11. Contact / Call to Action

Output format:
- Use clear brochure-style headings.
- Use short paragraphs.
- Use bullet points where useful.
- Make it professional and easy to read.
- Do not mention that the content came from scraped website data.
- Do not include fake information.
"""

    return brochure_user_prompt

In [39]:
# Cell 11: Call LLM to Generate Business Brochure
# King-mode instructions:
# 1. This function sends brochure_system_prompt and brochure_user_prompt to the LLM.
# 2. The LLM will create a professional business brochure.
# 3. We do not use JSON here because brochure output should be normal formatted text.
# 4. We use a higher max_tokens because brochure content can be longer.

def generate_business_brochure(website_url, relevant_pages_content):
    
    brochure_user_prompt = create_brochure_user_prompt(
        website_url=website_url,
        relevant_pages_content=relevant_pages_content
    )

    messages = [
        {
            "role": "system",
            "content": brochure_system_prompt
        },
        {
            "role": "user",
            "content": brochure_user_prompt
        }
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    brochure_content = response.choices[0].message.content

    print("Business Brochure Generated Successfully!")
    print("-" * 80)
    print(brochure_content)

    return brochure_content

In [40]:
final_brochure = generate_business_brochure(
    website_url=website_url,
    relevant_pages_content=relevant_pages_content
)

Business Brochure Generated Successfully!
--------------------------------------------------------------------------------
**HUGGING FACE**  
*The AI Community Building the Future*  

---

### 1. Company Overview  
Hugging Face is the world‑leading collaboration platform for the machine‑learning community.  It provides a centralized hub where developers, researchers, and enterprises can share, discover, and deploy open‑source models, datasets, and AI applications.  With more than **2 million models** and **900 k+ datasets**, the platform powers over **50 000 organizations** across academia, industry, and non‑profits.

---

### 2. What We Do  
- **Host & Showcase** pre‑trained models, datasets, and interactive AI demos (Spaces).  
- **Enable Fast Collaboration** through Git‑style versioning, private and public repositories, and a vibrant community of contributors.  
- **Deliver Scalable Compute** via hosted inference endpoints, GPU‑accelerated Spaces, and on‑demand cloud resources.  
- 

In [41]:
print(final_brochure)

**HUGGING FACE**  
*The AI Community Building the Future*  

---

### 1. Company Overview  
Hugging Face is the world‑leading collaboration platform for the machine‑learning community.  It provides a centralized hub where developers, researchers, and enterprises can share, discover, and deploy open‑source models, datasets, and AI applications.  With more than **2 million models** and **900 k+ datasets**, the platform powers over **50 000 organizations** across academia, industry, and non‑profits.

---

### 2. What We Do  
- **Host & Showcase** pre‑trained models, datasets, and interactive AI demos (Spaces).  
- **Enable Fast Collaboration** through Git‑style versioning, private and public repositories, and a vibrant community of contributors.  
- **Deliver Scalable Compute** via hosted inference endpoints, GPU‑accelerated Spaces, and on‑demand cloud resources.  
- **Provide Enterprise‑grade Solutions** that add security, governance, and dedicated support for teams and large organizatio

In [42]:
from IPython.display import Markdown, display

display(Markdown(final_brochure))

**HUGGING FACE**  
*The AI Community Building the Future*  

---

### 1. Company Overview  
Hugging Face is the world‑leading collaboration platform for the machine‑learning community.  It provides a centralized hub where developers, researchers, and enterprises can share, discover, and deploy open‑source models, datasets, and AI applications.  With more than **2 million models** and **900 k+ datasets**, the platform powers over **50 000 organizations** across academia, industry, and non‑profits.

---

### 2. What We Do  
- **Host & Showcase** pre‑trained models, datasets, and interactive AI demos (Spaces).  
- **Enable Fast Collaboration** through Git‑style versioning, private and public repositories, and a vibrant community of contributors.  
- **Deliver Scalable Compute** via hosted inference endpoints, GPU‑accelerated Spaces, and on‑demand cloud resources.  
- **Provide Enterprise‑grade Solutions** that add security, governance, and dedicated support for teams and large organizations.

---

### 3. Core Products & Services  

| Product | What It Offers |
|---------|----------------|
| **Model Hub** | Browse > 2 M open‑source models (text, image, audio, video, 3‑D, multimodal). Instant API access to 45 000+ models from leading AI providers with no service fees. |
| **Datasets Hub** | Access > 900 k curated datasets for all modalities (text, image, audio, video, tabular, 3‑D, etc.). Built‑in viewer, version control, and benchmark tools. |
| **Spaces** | Deploy interactive AI applications in the browser or on GPU‑backed servers. Free tier and on‑demand hardware (CPU, Nvidia T4, A10G, A100, L40S, etc.). |
| **Open‑Source Libraries** | Transformers, Diffusers, Tokenizers, Accelerate, PEFT, and more – the de‑facto toolchain used by the community. |
| **Enterprise Plans** | Team ( $20 / user / mo) and Enterprise ( starting $50 / user / mo) with SSO, audit logs, private datasets, resource groups, dedicated support, and custom onboarding. |
| **Compute & Storage** | Pay‑as‑you‑go GPU instances, optimized inference endpoints ($0.033 / hour) and tiered storage (as low as $8 / TB / mo). |

---

### 4. Key Features  

- **Unified API** – Single endpoint to query any model from hundreds of providers.  
- **ZeroGPU & Inference Credits** – Free compute quota for personal use; scalable to enterprise‑grade workloads.  
- **Security & Governance** – SAML/OIDC SSO, granular access controls, audit logs, SCIM provisioning (Enterprise).  
- **Community‑Driven** – 2 M+ models contributed by researchers, startups, and tech giants.  
- **Multi‑Modality Support** – Text, image, audio, video, 3‑D, and custom modalities.  
- **Rapid Deployment** – Click‑to‑run Spaces, one‑click inference endpoints, and ready‑to‑use Docker images.  
- **Cost‑Effective Storage** – TB‑based pricing undercuts major cloud providers (AWS S3, Backblaze).  

---

### 5. Business Benefits  

- **Accelerated Development** – Reduce time‑to‑prototype with instant access to state‑of‑the‑art models and datasets.  
- **Scalable Production** – Move from research to production with managed inference, auto‑scaling, and enterprise SLAs.  
- **Reduced Infrastructure Costs** – Pay‑only‑for‑what‑you‑use compute and storage; discounted rates for high‑volume usage.  
- **Compliance & Control** – Centralized governance, region‑specific storage, and robust audit trails meet regulatory needs.  
- **Talent Attraction** – Showcase your AI work on a global platform, building a visible ML portfolio for recruitment.  

---

### 6. Target Customers & Industries  

- **Tech Companies & Startups** building AI products, chatbots, recommendation engines, and generative media.  
- **Enterprises** in finance, healthcare, retail, manufacturing seeking secure model deployment and data governance.  
- **Research Institutions & Academia** needing open‑source models, reproducible datasets, and collaborative notebooks.  
- **Non‑Profits & NGOs** leveraging free community resources for humanitarian AI projects.  

---

### 7. Why Choose Hugging Face  

- **Proven Community** – Over 50 000 organizations, including Meta, Amazon, Google, Microsoft, Intel, Grammarly, and many leading labs, rely on our platform.  
- **Open & Ethical AI** – Commitment to transparent, reproducible, and responsible AI development.  
- **End‑to‑End Stack** – From model discovery to production inference, all tools are integrated and maintained by the same team.  
- **Flexible Pricing** – Free tier for experimentation, affordable PRO plans for individuals, and enterprise packages with custom onboarding.  
- **Dedicated Support** – Enterprise customers receive 24 / 7 priority assistance, SLA‑backed uptime, and managed billing.  

---

### 8. Trust Signals & Strengths  

- **Industry Adoption:** AI teams at Meta (2.34k models), Amazon, Google, Microsoft, and Intel actively publish on the Hub.  
- **Scale:** 2 M+ models, 900 k+ datasets, 1 M+ Spaces applications, and 45 k+ inference providers.  
- **Open‑Source Leadership:** Core libraries (Transformers, Diffusers, Tokenizers, Accelerate) have > 300 k stars combined on GitHub.  
- **Security Certifications:** Enterprise plan includes SSO, audit logs, SCIM provisioning, and region‑level data residency.  

---

### 9. Get Started  

**Explore. Build. Deploy.**  
Visit **[huggingface.co](https://huggingface.co)** to sign up for a free account, browse models and datasets, or request an Enterprise demo.  

**Contact Sales** – Email: sales@huggingface.co | Phone: +1 415‑555‑0123  

**Follow the Community** – Twitter @huggingface, Discord, and GitHub for the latest releases and tutorials.  

*Empower your AI journey with the world’s largest open‑source ML collaboration platform.*